# RAG: Retrieval-Augmented Generation 논문 재현

**Paper**: Lewis et al., *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*, NeurIPS 2020. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)

**목표**
- 논문의 두 변형 **RAG-Sequence** (Eq. 1) 와 **RAG-Token** (Eq. 2) 을 같은 질문에 대해 비교한다.
- **구성요소**: Retriever = DPR (Karpukhin et al., 2020), Generator = BART-large.
- **태스크**: Open-Domain QA — Natural Questions (NQ-open).
- **두 가지 모드**
  1. **Mini 모드 (학습용)**: 직접 만든 작은 코퍼스에 DPR + FAISS + BART 를 손으로 조립해서 RAG-Sequence / RAG-Token 의 수식이 코드에서 어떻게 작동하는지 확인한다.
  2. **Faithful 모드 (논문 재현)**: HuggingFace `wiki_dpr` (논문에서 쓴 Wikipedia 21M passages + 사전 계산된 DPR ctx 임베딩 + exact FAISS) 위에서 사전학습 체크포인트 `facebook/rag-sequence-nq` / `facebook/rag-token-nq` 로 추론하고 NQ-open dev 일부에 대해 **Exact Match** 를 측정한다.

**런타임 가정**
- Colab **A100 80GB** 인스턴스. wiki_dpr 의 exact FAISS 인덱스는 디스크 약 65GB + RAM 약 35GB 가 들어가므로 일반 T4 / 무료 인스턴스에서는 OOM 위험이 있다.
- A100 이 아니면 5장 (Faithful 모드) 의 셀들은 그냥 건너뛰거나 `index_name='compressed'` 로 바꿔서 실행한다.

## 0. 환경 설정

Colab 새 인스턴스에서 한 번만 실행하면 된다.

In [ ]:
!nvidia-smi 

In [ ]:
# wiki_dpr 의 FAISS 인덱스는 CPU 메모리에서 동작하고 (RagRetriever 가 CPU 로 검색),
# Mini 모드의 인덱스도 16개 패시지밖에 안 돼서 GPU FAISS 가 필요 없다.
# ──────────────────────────────────────────────────────────────────────
# 중요: faiss-cpu < 1.9 는 numpy<2 를 강제 핀하기 때문에, Colab 의 기본
# numpy 2.x 를 다운그레이드시켜 torch 등 numpy-2-로-컴파일된 다른 C 확장
# 들과 ABI 충돌이 난다 ("Expected 96 from C header, got 88 from PyObject").
# 그래서 faiss-cpu 는 1.9+ 로 잡아야 한다.
# ──────────────────────────────────────────────────────────────────────
!pip -q install --upgrade \
    transformers==4.44.2 \
    datasets==2.21.0 \
    "faiss-cpu>=1.9.0" \
    accelerate==0.34.2 \
    sentencepiece==0.2.0

# 설치가 끝나면 반드시 한 번 런타임을 재시작하고, 셀 4 부터 다시 실행한다.
# (Colab: "Runtime → Restart session", 또는 아래 줄의 주석을 풀어서 강제 재시작)
# import os; os.kill(os.getpid(), 9)
print('\\n설치 완료. 이제 Runtime → Restart session 후 셀 4 부터 실행하세요.')

In [ ]:
import os, json, math, time, gc, random, shutil, subprocess

# ──────────────────────────────────────────────────────────────────────
# 디스크 자동 라우팅: Colab 인스턴스에 따라 기본 / 디스크가 작은 경우가 있다
# (예: A100 80GB 인스턴스는 ~235GB 인데 wiki_dpr.compressed 가 train split
# 생성 중 한 번 60GB 이상 부풀어서 터지기 쉬움). Pro+ 에는 보통 /mnt 쪽에
# local-scratch (~368GB) 가 따로 붙어 있으니, 가장 여유 큰 마운트에 HF
# 캐시를 보내고 ~/.cache/huggingface 를 그쪽으로 심볼릭링크한다.
# ──────────────────────────────────────────────────────────────────────
def _setup_hf_cache(min_alt_free_gb: int = 80) -> str:
    home_cache = os.path.expanduser('~/.cache/huggingface')

    # (1) 이전 실행에서 부분 다운로드된 wiki_dpr 잔해 정리 (메인 디스크 회수).
    for sub in ('datasets/wiki_dpr', 'datasets/facebook___wiki_dpr',
                'datasets/downloads', 'datasets/downloads_tmp'):
        p = os.path.join(home_cache, sub)
        if os.path.exists(p) and not os.path.islink(p):
            print(f'  cleaning leftover  {p}')
            shutil.rmtree(p, ignore_errors=True)

    # (2) 가장 여유 큰 비-루트 마운트 찾기.
    try:
        out = subprocess.check_output(['df', '-B1', '--output=avail,target'],
                                      text=True).splitlines()[1:]
    except Exception as e:
        print(f'  df 실패 ({e}) — 기본 캐시 사용')
        return home_cache

    cand = []
    skip_prefixes = ('/dev', '/proc', '/sys', '/run', '/etc', '/boot',
                     '/var/lock', '/var/cache', '/var/lib', '/snap')
    for line in out:
        parts = line.strip().split()
        if len(parts) < 2:
            continue
        try:
            av = int(parts[0]); tgt = parts[1]
        except ValueError:
            continue
        if tgt == '/' or not tgt.startswith('/'):
            continue
        if tgt.startswith(skip_prefixes):
            continue
        cand.append((av, tgt))
    cand.sort(reverse=True)

    if not cand or cand[0][0] < min_alt_free_gb * 2**30:
        print(f'  ({min_alt_free_gb}GB+ 여유 있는 별도 마운트 없음 — 기본 캐시 사용)')
        return home_cache

    best_avail, best_tgt = cand[0]
    cache_root = os.path.join(best_tgt, 'hf_cache')
    os.makedirs(cache_root, exist_ok=True)
    print(f'  picked  {best_tgt}  ({best_avail/2**30:.1f} GB free)')
    print(f'  HF cache → {cache_root}')

    # (3) ~/.cache/huggingface 를 새 위치로 심볼릭링크 (HF lib 들이 hardcoded
    # 경로를 쓰는 일부 코드 경로까지 커버하기 위함; 그리고 HF_HOME 도 같이 셋).
    os.makedirs(os.path.dirname(home_cache), exist_ok=True)
    if os.path.islink(home_cache):
        os.unlink(home_cache)
    elif os.path.isdir(home_cache):
        # 메인 캐시에 남은 작은 파일들 (모델 weight 등) 은 보존해서 새 위치로 옮긴다.
        for entry in os.listdir(home_cache):
            src = os.path.join(home_cache, entry)
            dst = os.path.join(cache_root, entry)
            if not os.path.exists(dst):
                try:
                    shutil.move(src, dst)
                except Exception:
                    pass
        shutil.rmtree(home_cache, ignore_errors=True)
    os.symlink(cache_root, home_cache)
    return cache_root


print('Setting up HF cache...')
HF_CACHE_ROOT = _setup_hf_cache()
os.environ['HF_HOME']                       = HF_CACHE_ROOT
os.environ['HF_DATASETS_CACHE']             = os.path.join(HF_CACHE_ROOT, 'datasets')
os.environ['HUGGINGFACE_HUB_CACHE']         = os.path.join(HF_CACHE_ROOT, 'hub')
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = '1'
print('  HF_HOME =', os.environ['HF_HOME'])

# 디스크 상황 한 줄 요약
subprocess.run(['df', '-h', '/', HF_CACHE_ROOT], check=False)

# ──────────────────────────────────────────────────────────────────────
# 표준 imports
# ──────────────────────────────────────────────────────────────────────
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
import torch
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_grad_enabled(False)  # 추론 전용 노트북
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print('device =', device)
print('torch =', torch.__version__, '| cuda =', torch.version.cuda)

## 1. RAG 개요

RAG 는 입력 $x$ 에 대해 검색기 $p_\eta(z\mid x)$ 로 top-$K$ 문서 $\{z_1,\dots,z_K\}$ 를 뽑고, 그 문서들을 조건으로 시퀀스 $y$ 를 생성한다. 두 변형은 **언제 marginalize 하는가** 가 다르다.

**RAG-Sequence (Eq. 1).** 한 시퀀스 전체에 대해 같은 문서를 사용한다.
$$p_{\text{RAG-Seq}}(y\mid x) \;\approx\; \sum_{z\in\text{top-}K} p_\eta(z\mid x)\, p_\theta(y\mid x, z) \;=\; \sum_{z\in\text{top-}K} p_\eta(z\mid x)\,\prod_{i=1}^{N} p_\theta(y_i\mid x, z, y_{<i}).$$
디코딩은 문서별로 beam search 를 돌리고, 후보 시퀀스들의 union 위에서 위 marginal 을 계산해 argmax 한다.

**RAG-Token (Eq. 2).** 토큰마다 다른 문서를 사용할 수 있다.
$$p_{\text{RAG-Tok}}(y\mid x) \;=\; \prod_{i=1}^{N}\; \sum_{z\in\text{top-}K} p_\eta(z\mid x)\, p_\theta(y_i\mid x, z, y_{<i}).$$
디코딩은 매 스텝에서 $K$ 개의 generator 분포를 검색 확률로 가중평균해서 다음 토큰의 marginalized 분포를 만든다.

**Retriever.** $p_\eta(z\mid x) \propto \exp\big(\mathbf{d}(z)^\top \mathbf{q}(x)\big)$ 의 dual-encoder 형태이고, 두 인코더는 DPR 의 BERT-base.

**Generator.** BART-large. 입력은 $x$ 와 $z$ 를 concat 한 토큰열.

## 2. 구성요소 1 — Retriever (DPR)

논문은 NQ / TriviaQA / WQ / CT 네 데이터셋으로 함께 학습한 DPR **multiset** 체크포인트를 사용한다 (`facebook/dpr-question_encoder-multiset-base`, `facebook/dpr-ctx_encoder-multiset-base`).

In [ ]:
from transformers import (
    DPRQuestionEncoder, DPRQuestionEncoderTokenizerFast,
    DPRContextEncoder, DPRContextEncoderTokenizerFast,
)

Q_ENC = 'facebook/dpr-question_encoder-multiset-base'
C_ENC = 'facebook/dpr-ctx_encoder-multiset-base'

q_tok = DPRQuestionEncoderTokenizerFast.from_pretrained(Q_ENC)
q_enc = DPRQuestionEncoder.from_pretrained(Q_ENC).to(device).eval()
c_tok = DPRContextEncoderTokenizerFast.from_pretrained(C_ENC)
c_enc = DPRContextEncoder.from_pretrained(C_ENC).to(device).eval()

def encode_questions(texts: List[str], batch_size: int = 16) -> np.ndarray:
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = q_tok(batch, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
        emb = q_enc(**enc).pooler_output
        out.append(emb.detach().cpu().numpy())
    return np.concatenate(out, axis=0).astype('float32')

def encode_contexts(passages: List[Dict[str, str]], batch_size: int = 16) -> np.ndarray:
    # 논문의 DPR 입력 포맷: [CLS] title [SEP] text [SEP].
    out = []
    for i in range(0, len(passages), batch_size):
        batch = passages[i:i+batch_size]
        enc = c_tok([p['title'] for p in batch],
                    [p['text']  for p in batch],
                    padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
        emb = c_enc(**enc).pooler_output
        out.append(emb.detach().cpu().numpy())
    return np.concatenate(out, axis=0).astype('float32')

# 동작 확인
_q = encode_questions(['Who wrote the novel 1984?'])
_c = encode_contexts([{'title': 'George Orwell', 'text': 'Eric Arthur Blair, known by his pen name George Orwell, wrote Nineteen Eighty-Four (1984), published in 1949.'}])
print('q emb', _q.shape, '| c emb', _c.shape, '| dot =', float(_q @ _c.T))

## 3. 구성요소 2 — Generator (BART-large)

원논문은 BART-large 를 generator 의 초기값으로 쓴다 (open-domain QA 의 경우 사전학습된 DPR + BART-large 를 NQ 등으로 함께 fine-tune). 여기서는 fine-tune 된 가중치를 직접 학습하지 않고, **Mini 모드** 에서는 raw BART-large 로 mechanics 를 보여주고, **Faithful 모드** 에서는 NQ 로 fine-tune 된 `facebook/rag-sequence-nq` / `facebook/rag-token-nq` (DPR + BART-large) 를 그대로 사용한다.

In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizerFast

bart_name = 'facebook/bart-large'
bart_tok = BartTokenizerFast.from_pretrained(bart_name)
bart = BartForConditionalGeneration.from_pretrained(bart_name).to(device).eval()
print('BART loaded:', sum(p.numel() for p in bart.parameters())/1e6, 'M params')

## 4. Mini 모드 — RAG 를 손으로 조립

여기서는 작은 코퍼스 (수십 개의 위키 패시지) 위에서 RAG-Sequence 와 RAG-Token 의 디코딩을 **HF 의 RAG 클래스에 의존하지 않고** 직접 구현한다. 목표는 두 수식이 정확히 어떻게 코드가 되는지 보는 것이다.

### 4.1 작은 코퍼스 만들기

샘플 NQ 질문 몇 개에 답할 수 있도록 손으로 큐레이션한 짧은 위키 발췌. 실험 재현에는 충분하지 않고 mechanics 확인용이다.

In [ ]:
mini_corpus = [
    {'title': 'George Orwell',           'text': 'Eric Arthur Blair, known by his pen name George Orwell, was an English novelist who wrote Nineteen Eighty-Four (1984), published in June 1949.'},
    {'title': 'Nineteen Eighty-Four',    'text': 'Nineteen Eighty-Four is a dystopian social science fiction novel by English novelist George Orwell, published on 8 June 1949.'},
    {'title': 'Animal Farm',             'text': 'Animal Farm is an allegorical novella by George Orwell, first published in England on 17 August 1945.'},
    {'title': 'Mount Everest',           'text': 'Mount Everest is Earth\u2019s highest mountain above sea level, located in the Mahalangur Himal sub-range of the Himalayas, with a peak at 8,848.86 m.'},
    {'title': 'K2',                      'text': 'K2, at 8,611 m above sea level, is the second-highest mountain on Earth, after Mount Everest, located on the China\u2013Pakistan border.'},
    {'title': 'Albert Einstein',         'text': 'Albert Einstein was a German-born theoretical physicist who developed the theory of relativity and received the Nobel Prize in Physics in 1921 for the photoelectric effect.'},
    {'title': 'Theory of relativity',    'text': 'The theory of relativity usually encompasses two interrelated physics theories by Albert Einstein: special relativity and general relativity, proposed in 1905 and 1915.'},
    {'title': 'Marie Curie',             'text': 'Marie Sk\u0142odowska\u2013Curie was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity and won Nobel Prizes in Physics (1903) and Chemistry (1911).'},
    {'title': 'Apollo 11',               'text': 'Apollo 11 was the American spaceflight that first landed humans on the Moon. Commander Neil Armstrong and lunar module pilot Buzz Aldrin landed the Apollo Lunar Module Eagle on July 20, 1969.'},
    {'title': 'Neil Armstrong',          'text': 'Neil Alden Armstrong was an American astronaut and the first person to walk on the Moon on July 20, 1969, during the Apollo 11 mission.'},
    {'title': 'Eiffel Tower',            'text': 'The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower, completed in 1889.'},
    {'title': 'Paris',                   'text': 'Paris is the capital and most populous city of France, situated on the river Seine, with an estimated population of over 2 million.'},
    {'title': 'Python (programming language)', 'text': 'Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability. It was created by Guido van Rossum and first released in 1991.'},
    {'title': 'Guido van Rossum',        'text': 'Guido van Rossum is a Dutch programmer best known as the creator of the Python programming language, for which he was the \"benevolent dictator for life\" (BDFL) until 2018.'},
    {'title': 'World War II',            'text': 'World War II or the Second World War was a global conflict that lasted from 1939 to 1945, involving the vast majority of the world\u2019s countries.'},
    {'title': 'Mona Lisa',               'text': 'The Mona Lisa is a half-length portrait painting by Italian artist Leonardo da Vinci, considered an archetypal masterpiece of the Italian Renaissance, painted between 1503 and 1519.'},
]
print(len(mini_corpus), 'passages')

In [ ]:
import faiss

ctx_emb = encode_contexts(mini_corpus)
d = ctx_emb.shape[1]
index_mini = faiss.IndexFlatIP(d)         # 논문도 exact dot-product (FAISS IndexFlatIP)
index_mini.add(ctx_emb)
print('FAISS index ntotal =', index_mini.ntotal, 'dim =', d)

def retrieve_mini(question: str, k: int = 5):
    """Return (docs, scores, probs) — probs = softmax(scores) over the top-k."""
    q = encode_questions([question])
    scores, idx = index_mini.search(q, k)
    scores = scores[0]; idx = idx[0]
    docs = [mini_corpus[i] for i in idx]
    # p_eta(z | x) ∝ exp(d(z)·q(x))  →  softmax over the top-k
    probs = np.exp(scores - scores.max()); probs = probs / probs.sum()
    return docs, scores, probs

docs, s, p = retrieve_mini('Who wrote the novel 1984?', k=5)
for di, (doc, ss, pp) in enumerate(zip(docs, s, p)):
    print(f'[{di}] p={pp:.3f}  score={ss:.2f}  {doc["title"]}: {doc["text"][:90]}...')

### 4.2 RAG 의 generator 입력 포맷

HF `RagRetriever.postprocess_docs` 가 만드는 형식 (RagConfig 기본값) 그대로 따른다:

```
<title> / <text> // <question>
```

`title_sep=' / '`, `doc_sep=' // '`. 사전학습 RAG 체크포인트 (`rag-sequence-nq`, `rag-token-nq`) 가 이 포맷으로 학습됐기 때문에 Mini 모드도 동일하게 둔다.

In [ ]:
def format_input(question: str, doc: Dict[str, str]) -> str:
    # HF RagRetriever.postprocess_docs 가 만드는 실제 포맷.
    # RagConfig 기본값: title_sep=' / ', doc_sep=' // ', 그리고 순서는 title → text → question.
    return f"{doc['title']} / {doc['text']} // {question}"

print(format_input('Who wrote 1984?', mini_corpus[0]))

### 4.3 RAG-Sequence 디코딩 (Eq. 1)

**알고리즘 ("Thorough" decoding, 논문 §2.3)**
1. top-$K$ 문서 각각에 대해 generator 를 돌려 beam 후보 시퀀스 집합 $Y_z$ 를 모은다.
2. 모든 후보의 합집합 $Y = \bigcup_z Y_z$ 에 대해, 각 $y\in Y$ 의 marginal 점수를 계산한다:
$$\log p(y\mid x) \;=\; \log\sum_{z}\, p_\eta(z\mid x)\, p_\theta(y\mid x,z).$$
   ($y$ 가 어떤 beam 에서 나오지 않은 문서 $z$ 에 대해서도 그 $z$ 로 $p_\theta(y\mid x,z)$ 를 다시 평가해야 정확.)
3. argmax 한다.

구현 편의상 BART-large (NQ 로 fine-tune 되지 않음) 으로는 정답 생성이 잘 안 될 수 있다. 그래서 5장에서 fine-tune 된 RAG 체크포인트로 동일한 방식을 다시 돌릴 것이다.

In [ ]:
def _score_y_given_xz(question: str, doc: Dict[str, str], y_ids: torch.Tensor,
                      model=bart, tok=bart_tok) -> float:
    """log p_theta(y | x, z) — y_ids 는 generate() 가 돌려준 token id 시퀀스에서
    leading decoder_start 를 떼어낸 [<s>, t1, ..., </s>] 형태."""
    enc = tok(format_input(question, doc), return_tensors='pt', truncation=True, max_length=512).to(device)
    pad_id = tok.pad_token_id
    labels = y_ids.clone().to(device)
    # pad 위치는 loss 에서 제외 (-100 = CrossEntropyLoss ignore_index).
    labels_masked = labels.clone()
    labels_masked[labels_masked == pad_id] = -100
    out = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
                labels=labels_masked.unsqueeze(0))
    n_tokens = int((labels != pad_id).sum().item())
    if n_tokens == 0:
        return 0.0
    # out.loss 는 non-ignored 토큰에 대한 평균 NLL.  sum log p = -loss * n_tokens.
    return -float(out.loss) * n_tokens

def rag_sequence_generate(question: str, k: int = 5, num_beams: int = 4, max_new_tokens: int = 20,
                          model=bart, tok=bart_tok, retrieve_fn=retrieve_mini) -> Tuple[str, List[Tuple[str, float]]]:
    docs, scores, probs = retrieve_fn(question, k=k)
    log_p_z = np.log(probs + 1e-12)
    decoder_start = model.config.decoder_start_token_id

    # 1) 문서별 beam search — token id 시퀀스 그대로 보관 (text round-trip 회피).
    candidate_ids: Dict[Tuple[int, ...], torch.Tensor] = {}
    for doc in docs:
        enc = tok(format_input(question, doc), return_tensors='pt', truncation=True, max_length=512).to(device)
        gen = model.generate(**enc, num_beams=num_beams, num_return_sequences=num_beams,
                             max_new_tokens=max_new_tokens, early_stopping=True)
        for g in gen:
            seq = g.detach().cpu()
            # generate 출력은 [decoder_start, <s>, t1, ..., </s>, (pad...)]. labels 로 쓰려면 leading
            # decoder_start 만 제거하고 그 뒤를 그대로 target 으로 사용한다.
            if seq.numel() > 0 and seq[0].item() == decoder_start:
                seq = seq[1:]
            if seq.numel() == 0:
                continue
            candidate_ids[tuple(seq.tolist())] = seq

    # 2) 각 후보 y 에 대해 모든 K 개 문서로 marginal log-prob 계산 (Thorough decoding).
    ranked: List[Tuple[str, float]] = []
    for _, y_ids in candidate_ids.items():
        per_doc = np.empty(len(docs), dtype=np.float64)
        for di, (z, lpz) in enumerate(zip(docs, log_p_z)):
            per_doc[di] = lpz + _score_y_given_xz(question, z, y_ids, model=model, tok=tok)
        # log sum_z p_eta(z|x) p_theta(y|x,z)  via logsumexp.
        m = per_doc.max()
        marg = float(m + np.log(np.exp(per_doc - m).sum()))
        y_txt = tok.decode(y_ids, skip_special_tokens=True).strip()
        ranked.append((y_txt, marg))

    ranked.sort(key=lambda t: t[1], reverse=True)
    return (ranked[0][0] if ranked else ''), ranked[:10]

_ans, _rank = rag_sequence_generate('Who wrote the novel 1984?', k=5, num_beams=4, max_new_tokens=10)
print('RAG-Sequence (raw BART) →', _ans)
for y, s in _rank[:5]:
    print(f'  log p(y|x)={s:.2f}  {y!r}')

### 4.4 RAG-Token 디코딩 (Eq. 2)

**알고리즘**: beam search 인데, 각 스텝에서 next-token 분포가 $K$ 개의 generator 분포의 검색-확률 가중평균이다.
$$p(y_i\mid x, y_{<i}) \;=\; \sum_{z} p_\eta(z\mid x)\, p_\theta(y_i\mid x, z, y_{<i}).$$
구현은 $K$ 개의 encoder hidden state 을 미리 만들어 두고, 디코더를 $K$ 번 평행 호출해 logits 을 평균하면 된다.

In [ ]:
from transformers.modeling_outputs import BaseModelOutput

def rag_token_generate(question: str, k: int = 5, num_beams: int = 4, max_new_tokens: int = 20,
                       model=bart, tok=bart_tok, retrieve_fn=retrieve_mini) -> str:
    docs, scores, probs = retrieve_fn(question, k=k)
    log_p_z = torch.tensor(np.log(probs + 1e-12), dtype=torch.float32, device=device)  # [K]

    # K 개의 encoder 상태 미리 계산.
    enc_inputs = tok([format_input(question, d) for d in docs], return_tensors='pt',
                     padding=True, truncation=True, max_length=512).to(device)
    enc_out = model.get_encoder()(input_ids=enc_inputs['input_ids'], attention_mask=enc_inputs['attention_mask'])
    enc_hid = enc_out.last_hidden_state                       # [K, S, H]
    enc_mask = enc_inputs['attention_mask']                   # [K, S]
    K = enc_hid.size(0)

    bos = model.config.decoder_start_token_id
    eos = model.config.eos_token_id
    pad = tok.pad_token_id

    # beam 상태: [(tokens: List[int], logprob_sum: float, finished: bool)]
    beams = [([bos], 0.0, False)]

    for step in range(max_new_tokens):
        if all(b[2] for b in beams):
            break
        new_candidates = []
        for tokens, lp, fin in beams:
            if fin:
                new_candidates.append((tokens, lp, True))
                continue
            dec_in = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0).expand(K, -1)  # [K, T]
            dec_out = model(
                encoder_outputs=BaseModelOutput(last_hidden_state=enc_hid),
                attention_mask=enc_mask,
                decoder_input_ids=dec_in,
                use_cache=False,
            )
            logits = dec_out.logits[:, -1, :]                  # [K, V]
            log_p_y_given_xz = F.log_softmax(logits, dim=-1)   # [K, V]
            # marginalize over docs:  log sum_z exp( log p_eta(z|x) + log p_theta(y_i | x, z, y_<i) )
            marg = torch.logsumexp(log_p_z.unsqueeze(-1) + log_p_y_given_xz, dim=0)  # [V]
            top = torch.topk(marg, num_beams)
            for tok_id, tok_lp in zip(top.indices.tolist(), top.values.tolist()):
                new_tokens = tokens + [tok_id]
                new_lp = lp + tok_lp
                new_fin = (tok_id == eos)
                new_candidates.append((new_tokens, new_lp, new_fin))
        # length-normalized 로 가지치기 (BOS 토큰은 길이에서 제외).
        new_candidates.sort(key=lambda b: b[1] / max(1, len(b[0]) - 1), reverse=True)
        beams = new_candidates[:num_beams]

    best = max(beams, key=lambda b: b[1] / max(1, len(b[0]) - 1))
    out_ids = [t for t in best[0] if t not in {bos, eos, pad}]
    return tok.decode(out_ids, skip_special_tokens=True).strip()

print('RAG-Token (raw BART) →', rag_token_generate('Who wrote the novel 1984?', k=5, num_beams=4, max_new_tokens=10))

### 4.5 같은 질문에 대해 두 변형 비교 (raw BART)

BART-large 는 NQ 로 fine-tune 되어 있지 않기 때문에, 출력이 답 자체보다는 "질문 + 답" 형태이거나 다소 장황할 수 있다. **수식이 코드로 어떻게 구현되는지를 보는 셀** 이고, 실제 reproduction 의 정량 지표는 5장에서 fine-tune 된 RAG 체크포인트로 측정한다.

In [ ]:
questions = [
    'Who wrote the novel 1984?',
    'What is the highest mountain on Earth?',
    'Who created the Python programming language?',
    'Who was the first person to walk on the Moon?',
    'Who painted the Mona Lisa?',
]
for q in questions:
    seq, _ = rag_sequence_generate(q, k=5, num_beams=4, max_new_tokens=12)
    tok_ans = rag_token_generate(q, k=5, num_beams=4, max_new_tokens=12)
    print('Q:', q)
    print('  RAG-Sequence:', seq)
    print('  RAG-Token   :', tok_ans)
    print()

## 5. Faithful 모드 — wiki_dpr + 사전학습 RAG

여기부터는 논문 셋업에 더 가깝게:
- **지식 소스**: `wiki_dpr` (DPR 논문에서 만든 Wikipedia psgs_w100 — 21M passages, 각 ~100 단어, DPR ctx 임베딩 사전계산).
- **FAISS 인덱스**: `compressed` (OPQ 양자화, RAM ~30GB). `exact` (IndexFlatIP, 논문이 쓴 형식과 정확히 동일, RAM ~65GB) 로 바꾸려면 아래 `INDEX_NAME` 만 바꿔주면 된다.
- **모델**: `facebook/rag-sequence-nq`, `facebook/rag-token-nq` — DPR question encoder + BART-large generator 가 NQ 로 함께 fine-tune 된 체크포인트.

### 메모리 주의 (중요)

`RagRetriever.from_pretrained(..., index_name='exact')` 는 wiki_dpr 인덱스를 통째로 시스템 RAM 에 올린다. 두 모델 (`rag-sequence-nq`, `rag-token-nq`) 마다 **별도 retriever** 를 만들면 같은 인덱스가 RAM 에 2번 복사돼 80GB 인스턴스에서도 OOM 이 난다. 그래서 아래는 **retriever 하나를 두 모델이 공유** 하는 패턴을 쓴다 (두 체크포인트가 동일한 DPR question encoder 를 사용하므로 안전).

또 datasets ≥ 2.20 에서는 `wiki_dpr` 같은 script-based dataset 을 로드할 때 `trust_remote_code=True` 가 필요한데, `RagRetriever` 의 내부 `load_dataset` 호출이 이 플래그를 forward 하지 않으므로 **환경변수** 로 켠다.

### 첫 다운로드 시간

- `compressed`: ~30GB. 다운로드 약 30분~1시간.
- `exact`: ~70GB. 다운로드 1~2시간 + 인덱스 적재 추가 시간.

Colab 세션 디스크는 ~166GB 이므로 둘 다 들어가지만, 세션이 끊기면 다시 받아야 하니 가능하면 `compressed` 로 먼저 동작 확인 후 `exact` 로 바꾸길 권장.

In [ ]:
import datasets as hfds

# 'exact' 로 바꾸면 논문 형식과 정확히 일치하지만 RAM ~65GB + 다운로드 ~70GB 가 필요.
# 'compressed' (~30GB) 권장. 셀 4 에서 HF_HOME 을 큰 스크래치 디스크로 라우팅해뒀으므로
# 어느 쪽이든 디스크 OOM 없이 동작한다.
INDEX_NAME = 'compressed'

# wiki_dpr 의 사전 load 는 하지 않는다. 셀 24 의 RagRetriever 가 내부에서 한 번만 로드하고
# 두 RAG 모델이 그 retriever 를 공유 → 인덱스가 RAM 에 1 copy 만 남는다.
print('INDEX_NAME =', INDEX_NAME)
print('HF_HOME    =', os.environ.get('HF_HOME'))

In [ ]:
from transformers import RagRetriever, RagTokenizer, RagSequenceForGeneration, RagTokenForGeneration

RAG_SEQ = 'facebook/rag-sequence-nq'
RAG_TOK = 'facebook/rag-token-nq'

# Retriever 를 한 번만 로드해서 두 모델이 공유 → wiki_dpr 인덱스가 RAM 에 1 copy.
# rag-sequence-nq 와 rag-token-nq 는 같은 DPR question encoder 를 쓰므로 공유 안전.
retriever = RagRetriever.from_pretrained(RAG_SEQ, index_name=INDEX_NAME, use_dummy_dataset=False)

rag_seq_tok = RagTokenizer.from_pretrained(RAG_SEQ)
rag_tok_tok = RagTokenizer.from_pretrained(RAG_TOK)

rag_seq_model = RagSequenceForGeneration.from_pretrained(RAG_SEQ, retriever=retriever).to(device).eval()
rag_tok_model = RagTokenForGeneration  .from_pretrained(RAG_TOK, retriever=retriever).to(device).eval()
print('pretrained RAG models loaded (shared retriever)')

In [ ]:
@torch.no_grad()
def rag_answer(model, tokenizer, question: str, n_docs: int = 5, num_beams: int = 4, max_new_tokens: int = 20) -> str:
    enc = tokenizer.question_encoder(question, return_tensors='pt', truncation=True, max_length=128).to(device)
    gen = model.generate(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
                         num_beams=num_beams, num_return_sequences=1,
                         max_new_tokens=max_new_tokens, n_docs=n_docs)
    return tokenizer.batch_decode(gen, skip_special_tokens=True)[0].strip()

for q in questions:
    seq_ans = rag_answer(rag_seq_model, rag_seq_tok, q)
    tok_ans = rag_answer(rag_tok_model, rag_tok_tok, q)
    print('Q:', q)
    print('  pretrained RAG-Sequence:', seq_ans)
    print('  pretrained RAG-Token   :', tok_ans)
    print()

## 6. Natural Questions (NQ-open) 일부에서 정량 평가

논문 Table 1 — open-domain QA exact-match.

| Model | NQ | TQA | WQ | CT |
|---|---|---|---|---|
| Closed Book T5-11B + SSM | 36.6 | 60.5 | 44.7 | – |
| REALM | 40.4 | – | 40.7 | 46.8 |
| DPR | 41.5 | 57.9 | 41.1 | 50.6 |
| **RAG-Token** | 44.1 | 55.2 | 45.5 | 50.0 |
| **RAG-Sequence** | **44.5** | **56.8** | **45.2** | **52.2** |

여기서는 NQ-open dev set 의 작은 슬라이스 (예: 처음 500개) 로 RAG-Sequence / RAG-Token 의 EM 을 측정해 위 수치 부근으로 떨어지는지 본다.

In [ ]:
import re, string, unicodedata

def normalize_answer(s: str) -> str:
    # NQ / SQuAD 표준 normalization + 액센트/유니코드 변형 흡수.
    s = unicodedata.normalize('NFD', s)
    s = ''.join(ch for ch in s if unicodedata.category(ch) != 'Mn')   # combining marks 제거
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ' '.join(s.split())
    return s

def exact_match(pred: str, golds: List[str]) -> int:
    p = normalize_answer(pred)
    return int(any(p == normalize_answer(g) for g in golds))

In [ ]:
# NQ-open: 질문 + (여러) 정답 리스트. HF dataset 'nq_open'.
nq = hfds.load_dataset('nq_open', split='validation')
print(nq)
print(nq[0])

In [ ]:
N_EVAL = 500    # 시간을 보고 늘리거나 줄이세요. 전체 dev = 3,610.
subset = nq.select(range(N_EVAL))

preds_seq, preds_tok, golds_all = [], [], []
t0 = time.time()
for i, ex in enumerate(subset):
    q = ex['question']
    a_seq = rag_answer(rag_seq_model, rag_seq_tok, q, n_docs=5, num_beams=4)
    a_tok = rag_answer(rag_tok_model, rag_tok_tok, q, n_docs=5, num_beams=4)
    preds_seq.append(a_seq); preds_tok.append(a_tok); golds_all.append(ex['answer'])
    if (i+1) % 25 == 0:
        em_s = np.mean([exact_match(p, g) for p, g in zip(preds_seq, golds_all)])
        em_t = np.mean([exact_match(p, g) for p, g in zip(preds_tok, golds_all)])
        print(f'[{i+1}/{N_EVAL}] elapsed {time.time()-t0:.0f}s   EM seq={em_s*100:.2f}  EM tok={em_t*100:.2f}')

em_seq = np.mean([exact_match(p, g) for p, g in zip(preds_seq, golds_all)])
em_tok = np.mean([exact_match(p, g) for p, g in zip(preds_tok, golds_all)])
print(f'\nFINAL  EM RAG-Sequence = {em_seq*100:.2f}    EM RAG-Token = {em_tok*100:.2f}    (논문 보고: 44.5 / 44.1)')

In [ ]:
# 틀린/맞은 사례 몇 개 살펴보기.
print('=== 같은 질문에서 두 변형이 다르게 답한 사례 ===')
shown = 0
for q_ex, ps, pt, golds in zip(subset, preds_seq, preds_tok, golds_all):
    if normalize_answer(ps) != normalize_answer(pt):
        es = exact_match(ps, golds); et = exact_match(pt, golds)
        print(f'Q: {q_ex["question"]}')
        print(f'  gold       : {golds}')
        print(f'  RAG-Seq    : {ps!r}   EM={es}')
        print(f'  RAG-Tok    : {pt!r}   EM={et}')
        print()
        shown += 1
        if shown >= 8:
            break

### 6.1 결과 시각화

측정한 EM 을 논문 Table 1 의 NQ 결과들과 한 테이블에 모으고, 막대그래프로 비교한다. 그리고 우리 RAG-Sequence / RAG-Token 의 샘플별 예측을 별도 테이블로 본다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ─── Table A: NQ-open EM 비교 (논문 Table 1 의 NQ 열 + 이번 실험에서 측정한 두 값) ───
em_table = pd.DataFrame([
    {'Model': 'Closed-Book T5-11B+SSM',     'NQ EM (paper)': 36.6, 'NQ EM (this run)': None},
    {'Model': 'REALM',                       'NQ EM (paper)': 40.4, 'NQ EM (this run)': None},
    {'Model': 'DPR (extractive)',            'NQ EM (paper)': 41.5, 'NQ EM (this run)': None},
    {'Model': 'RAG-Token (facebook/rag-token-nq)',       'NQ EM (paper)': 44.1, 'NQ EM (this run)': round(em_tok * 100, 2)},
    {'Model': 'RAG-Sequence (facebook/rag-sequence-nq)', 'NQ EM (paper)': 44.5, 'NQ EM (this run)': round(em_seq * 100, 2)},
])
em_table['Δ vs paper'] = (em_table['NQ EM (this run)'] - em_table['NQ EM (paper)']).round(2)

print(f'NQ-open dev subset size: {len(preds_seq)} questions  (full dev = 3,610)')
print()
display(em_table)

# ─── Bar chart: paper vs this-run for the same models ───
fig, ax = plt.subplots(figsize=(8.5, 4.5))
x = np.arange(len(em_table))
w = 0.38
ax.bar(x - w/2, em_table['NQ EM (paper)'].fillna(0).values, width=w, label='Paper', color='#9aa0a6')
ax.bar(x + w/2, em_table['NQ EM (this run)'].fillna(0).values, width=w, label=f'This run (n={len(preds_seq)})', color='#1a73e8')
ax.set_xticks(x)
ax.set_xticklabels([m.split(' (')[0] for m in em_table['Model']], rotation=20, ha='right')
ax.set_ylabel('Exact Match (NQ-open dev, %)')
ax.set_title('RAG vs baselines — Natural Questions open-domain QA')
ax.set_ylim(0, max(em_table['NQ EM (paper)'].max(), em_table['NQ EM (this run)'].fillna(0).max()) + 5)
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='lower right')
for i, v in enumerate(em_table['NQ EM (paper)'].fillna(0).values):
    ax.text(i - w/2, v + 0.3, f'{v:.1f}', ha='center', fontsize=8, color='#5f6368')
for i, v in enumerate(em_table['NQ EM (this run)'].fillna(0).values):
    if v > 0:
        ax.text(i + w/2, v + 0.3, f'{v:.1f}', ha='center', fontsize=8, color='#1a73e8')
plt.tight_layout()
plt.show()

# ─── Table B: per-question predictions (Seq vs Tok), 처음 N 개 ───
n_show = min(30, len(preds_seq))
rows = []
for i in range(n_show):
    q = subset[i]['question']
    g = subset[i]['answer']
    ps, pt = preds_seq[i], preds_tok[i]
    rows.append({
        'Q': q if len(q) < 70 else q[:67] + '...',
        'Gold': ' | '.join(g)[:50],
        'RAG-Seq': ps[:40],
        'EM-Seq': '✓' if exact_match(ps, g) else '·',
        'RAG-Tok': pt[:40],
        'EM-Tok': '✓' if exact_match(pt, g) else '·',
    })
sample_table = pd.DataFrame(rows)
print(f'\n샘플별 예측 (처음 {n_show}개):')
display(sample_table)

# ─── 요약: 두 변형의 동의/불일치 분포 ───
def normed_eq(a, b): return normalize_answer(a) == normalize_answer(b)
agree = sum(normed_eq(s, t) for s, t in zip(preds_seq, preds_tok))
both_right   = sum(exact_match(s, g) and exact_match(t, g)             for s, t, g in zip(preds_seq, preds_tok, golds_all))
only_seq     = sum(exact_match(s, g) and not exact_match(t, g)         for s, t, g in zip(preds_seq, preds_tok, golds_all))
only_tok     = sum(not exact_match(s, g) and exact_match(t, g)         for s, t, g in zip(preds_seq, preds_tok, golds_all))
both_wrong   = sum(not exact_match(s, g) and not exact_match(t, g)     for s, t, g in zip(preds_seq, preds_tok, golds_all))
n = len(preds_seq)
summary = pd.DataFrame([
    {'metric': '두 변형이 같은 답',          'count': agree,      'pct': f'{agree/n*100:.1f}%'},
    {'metric': '둘 다 정답',                  'count': both_right, 'pct': f'{both_right/n*100:.1f}%'},
    {'metric': 'Sequence 만 정답',           'count': only_seq,   'pct': f'{only_seq/n*100:.1f}%'},
    {'metric': 'Token 만 정답',              'count': only_tok,   'pct': f'{only_tok/n*100:.1f}%'},
    {'metric': '둘 다 오답',                  'count': both_wrong, 'pct': f'{both_wrong/n*100:.1f}%'},
])
print('\nSequence vs Token 동의/불일치 분해:')
display(summary)

## 7. 정리

- **Mini 모드** 에서는 retriever (DPR + FAISS), generator (BART), 그리고 RAG-Sequence (Eq. 1) / RAG-Token (Eq. 2) 의 marginalization 디코딩을 직접 구현했다. raw BART 라서 생성 자체가 깔끔하진 않지만, 두 변형의 **수식 차이가 코드에서 어디로 가는지** — 시퀀스 단위 marginal 인지, 토큰 단위 marginal 인지 — 확인할 수 있다.
- **Faithful 모드** 에서는 wiki_dpr (논문이 쓴 21M-passage 인덱스) + 사전학습 RAG 체크포인트로 NQ-open dev 일부에서 EM 을 잰다. 논문 보고치 (44.5 / 44.1) 와의 차이는 평가 슬라이스 크기, num_beams, n_docs, normalize 함수의 작은 차이에서 생긴다.

**다음으로 확장해 볼 만한 것**
1. 같은 retriever 인덱스 위에서 **DPR-only baseline** (extractive reader, 예: `facebook/dpr-reader-multiset-base`) 를 돌려 Table 1 의 DPR 41.5 와 비교.
2. RAG-Sequence 의 **Thorough vs Fast decoding** 차이를 mini 모드에서 직접 측정 (논문 §2.3).
3. **n_docs** (top-K) 를 5 → 10 → 20 으로 늘려가며 EM 변화 — 논문 Figure 3 재현.
4. 자체 도메인 코퍼스 (예: 회사 위키) 위에서 같은 파이프라인을 돌려보기 — RAG 의 실제 효용은 여기 있다.